In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('updated_clubs.csv')

In [3]:
categories = df['category'].unique().tolist()

In [5]:
import json

with open("student_club_recommendation_examples.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

print(len(data))
print(data[0])

200
{'goal': 'I want to prepare for math competitions and sharpen my skills', 'interest': 'mathematics, puzzles, competitions', 'text': "I'm still figuring out campus life, but I know I'd enjoy a club where I can sharpen my math skills. Something centered on math problems and competitions sounds right.", 'categories': ['academic']}


In [6]:
X = []
for sample in data:
    combined_text = f"Goal: {sample['goal']} Interest: {sample['interest']} Text: {sample['text']}"
    X.append(combined_text)

print(X[0])

Goal: I want to prepare for math competitions and sharpen my skills Interest: mathematics, puzzles, competitions Text: I'm still figuring out campus life, but I know I'd enjoy a club where I can sharpen my math skills. Something centered on math problems and competitions sounds right.


In [7]:
label_list = ['cultural', 'academic', 'sports', 'professional', 'social', 'art']
y = []
for sample in data:
    label_vector = [1 if label in sample["categories"] else 0 for label in label_list]
    y.append(label_vector)
y = np.array(y)

print(y[0])

[0 1 0 0 0 0]


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000)
    ))
])

logreg_model.fit(X_train, y_train)
print("Baseline model trained.")

Baseline model trained.


In [30]:
from sklearn.metrics import f1_score, classification_report

y_pred_logreg = logreg_model.predict(X_test)

print("Logistic Regression Results")
print("Micro F1:", f1_score(y_test, y_pred_logreg, average="micro"))
print("Macro F1:", f1_score(y_test, y_pred_logreg, average="macro"))
print()
print(classification_report(y_test, y_pred_logreg, target_names=label_list))

Logistic Regression Results
Micro F1: 0.616822429906542
Macro F1: 0.6285129616398967

              precision    recall  f1-score   support

    cultural       1.00      0.44      0.62        18
    academic       1.00      0.29      0.44        14
      sports       1.00      0.70      0.82        10
professional       1.00      0.67      0.80         9
      social       1.00      0.27      0.42        15
         art       1.00      0.50      0.67         8

   micro avg       1.00      0.45      0.62        74
   macro avg       1.00      0.48      0.63        74
weighted avg       1.00      0.45      0.60        74
 samples avg       0.80      0.59      0.65        74



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
examples = [
    "Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.",
    "Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.",
    "Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events."
]

y_prob = logreg_model.predict_proba(examples)
threshold = 0.3
preds = (y_prob >= threshold).astype(int)

for text, probs, pred in zip(examples, y_prob, preds):
    print("TEXT:", text)

    ranked = sorted(zip(label_list, probs), key=lambda x: x[1], reverse=True)
    print("Scores:")
    for label, score in ranked:
        print(f"  {label}: {score:.3f}")

    labels = [label_list[i] for i in range(len(label_list)) if pred[i] == 1]
    print("Predicted:", labels)
    print("-" * 60)

TEXT: Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.
Scores:
  art: 0.341
  academic: 0.319
  sports: 0.275
  professional: 0.256
  social: 0.226
  cultural: 0.214
Predicted: ['academic', 'art']
------------------------------------------------------------
TEXT: Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.
Scores:
  sports: 0.558
  art: 0.294
  professional: 0.215
  cultural: 0.209
  social: 0.205
  academic: 0.204
Predicted: ['sports']
------------------------------------------------------------
TEXT: Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events.
Scores:
  social: 0.655
  art: 0.276
  professional: 0.271
  sports: 0.227
  cultural: 0.198
  academic: 0.194
Predicted: ['social']
------------------------------------------------------------


In [32]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier

models = {
    "LogisticRegression": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )),
        ("clf", OneVsRestClassifier(
            LogisticRegression(max_iter=2000)
        ))
    ]),

    "LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )),
        ("clf", OneVsRestClassifier(
            LinearSVC()
        ))
    ]),

    "MultinomialNB": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )),
        ("clf", OneVsRestClassifier(
            MultinomialNB()
        ))
    ]),

    "SGDClassifier": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )),
        ("clf", OneVsRestClassifier(
            SGDClassifier(loss="log_loss", random_state=42)
        ))
    ])
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    micro_f1 = f1_score(y_test, y_pred, average="micro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    results[name] = {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1
    }

    print("=" * 70)
    print(name)
    print("Micro F1:", micro_f1)
    print("Macro F1:", macro_f1)

LogisticRegression
Micro F1: 0.616822429906542
Macro F1: 0.6285129616398967
LinearSVC
Micro F1: 0.9361702127659575
Macro F1: 0.933591871091871
MultinomialNB
Micro F1: 0.7804878048780488
Macro F1: 0.7746326693195765
SGDClassifier
Micro F1: 0.9361702127659575
Macro F1: 0.933591871091871


In [52]:
examples = [
    "Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.",
    "Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.",
    "Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events.",
    "Goal: test yourself as an actor Interest: theater Text: i want kazakh  art ",
    "i hate academia"
]

MODEL_NAME = "LogisticRegression"

model = models[MODEL_NAME]
model.fit(X_train, y_train)

# preds = model.predict(examples)

# print(label_list)
# print(preds)
y_prob = model.predict_proba(examples)
threshold = 0.3
preds = (y_prob >= threshold).astype(int)

for text, pred in zip(examples, preds):
    print("TEXT:", text)

    # ranked = sorted(zip(label_list, preds), key=lambda x: x[1], reverse=True)
    print("Scores:")
    for label, score in ranked:
        print(f"  {label}: {score:.3f}")

    labels = [label_list[i] for i in range(len(label_list)) if pred[i] == 1]
    print("Predicted:", labels)
    print("-" * 60)

TEXT: Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.
Scores:
  social: 0.655
  art: 0.276
  professional: 0.271
  sports: 0.227
  cultural: 0.198
  academic: 0.194
Predicted: ['cultural']
------------------------------------------------------------
TEXT: Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.
Scores:
  social: 0.655
  art: 0.276
  professional: 0.271
  sports: 0.227
  cultural: 0.198
  academic: 0.194
Predicted: ['cultural']
------------------------------------------------------------
TEXT: Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events.
Scores:
  social: 0.655
  art: 0.276
  professional: 0.271
  sports: 0.227
  cultural: 0.198
  academic: 0.194
Predicted: ['cultural']
------------------------------------------------------------
TEXT: Goal: test yourself as an actor Interest: theater Text: 

In [53]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the dataset
df = pd.read_csv('cleaned_data.csv')

# 2. Prepare features (X) and target (y)
# We use 'description' as the training text and 'category' as the label
X = df['description']
y = df['category']

# 3. Split the data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Create a Machine Learning Pipeline
# This automatically handles text vectorization and classification in one flow
# Add "student" and "club" to stop words to stop the bias
custom_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words()) + ['student', 'students', 'club', 'university', 'nu']

model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stop_words, max_features=5000)),
    ('clf', LogisticRegression(class_weight='balanced')) # 'balanced' helps with the class gap
])

# 5. Train the model
model_pipeline.fit(X_train, y_train)

# 6. Evaluate the model performance
y_pred = model_pipeline.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# 7. Function to predict the category for new user input
def predict_club_category(user_text):
    """
    Takes a string input and returns the predicted club category.
    """
    prediction = model_pipeline.predict([user_text])
    print("pred:")
    print(prediction)
    return prediction[0]

# --- EXAMPLE USAGE ---
user_input = "i hate academic clubs"
result = predict_club_category(user_input)
print(f"\nUser Input: '{user_input}'")
print(f"Predicted Category: {result}")

# 8. Save the model to a file
with open('club_classifier_model.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)

Model Accuracy: 79.17%

Detailed Classification Report:
              precision    recall  f1-score   support

    academic       0.75      1.00      0.86         9
         art       0.50      1.00      0.67         1
    cultural       1.00      0.80      0.89         5
professional       0.00      0.00      0.00         1
      social       0.00      0.00      0.00         3
      sports       0.83      1.00      0.91         5

    accuracy                           0.79        24
   macro avg       0.51      0.63      0.55        24
weighted avg       0.68      0.79      0.72        24

pred:
['academic']

User Input: 'i hate academic clubs'
Predicted Category: academic


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
